# 📈 Astoria Cleaning Services — Service Demand & Revenue Prediction
### Problem Statement: Predictive Revenue & Effort Modeling

**Objective:** Build a regression model to predict `order_value_sgd` based on order characteristics to:
- Forecast revenue based on incoming bookings.
- Understand the key drivers of order value.
- Optimize pricing and resource allocation strategies.

**Business Impact Expected:**
- More accurate financial forecasting.
- Identification of high-value service categories.
- Improved resource planning during peak demand periods.

---

## 📋 Data Reference — Columns Used 

| Column | Type | Purpose |
|--------|------|---------|
| `category` | Categorical | Type of items being cleaned |
| `service` | Categorical | Specific service performed |
| `quantity` | Numerical | Number of items in the order |
| `base_price_sgd` | Numerical | Starting price for the service |
| `delicate_surcharge` | Numerical | Additional cost for delicate items |
| `express_multiplier` | Numerical | Price multiplier for express services |
| `order_value_sgd` | Numerical | **Target Variable** (Total order revenue) |


## 1. Environment Setup & Data Loading

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Load dataset
df = pd.read_csv('../../data1/astoria_orders.csv')
print(f'Dataset loaded: {df.shape[0]} rows and {df.shape[1]} columns.')

Dataset loaded: 5000 rows and 27 columns.


## 2. Feature Selection & Preprocessing

In [4]:
# Select relevant features and target
features = ['category', 'service', 'quantity', 'base_price_sgd', 'delicate_surcharge', 'express_multiplier']
target = 'order_value_sgd'

# Drop missing values in the target if any
df = df.dropna(subset=[target])

# Split features and target
X = df[features]
y = df[target]

# Identify categorical and numerical columns
cat_cols = ['category', 'service']
num_cols = ['quantity', 'base_price_sgd', 'delicate_surcharge', 'express_multiplier']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Data split into training and testing sets.')

Data split into training and testing sets.


## 3. Model Training & Evaluation

In [5]:
# Define the model pipeline (Random Forest Regressor)
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'Model Performance:')
print(f'Mean Absolute Error: ${mae:.2f}')
print(f'Root Mean Squared Error: ${rmse:.2f}')
print(f'R^2 Score: {r2:.4f}')

Model Performance:
Mean Absolute Error: $2.67
Root Mean Squared Error: $13.57
R^2 Score: 0.9858


## 4. Visualizing Model Results

In [6]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Order Value ($)')
plt.ylabel('Predicted Order Value ($)')
plt.title('Actual vs Predicted Order Value')
plt.savefig('actual_vs_predicted.png')
plt.close()

# Feature Importance
regressor = model.named_steps['regressor']
onehot_cols = model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_features = num_cols + list(onehot_cols)
importances = regressor.feature_importances_
indices = np.argsort(importances)[-10:] # Top 10 features

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [all_features[i] for i in indices])
plt.xlabel('Feature Importance')
plt.title('Top 10 Drivers of Order Value')
plt.savefig('feature_importance.png')
plt.close()

## 5. Strategic Recommendations

Based on the model performance and feature importance:

1. **Dynamic Pricing Strategy:** The `express_multiplier` and `base_price_sgd` are major drivers of order value. Astoria can optimize their express tiers to maximize revenue.
2. **Capacity Management:** By predicting the expected `order_value_sgd` from bookings, Astoria can better allocate high-skilled staff to high-value orders (e.g., Traditional or delicate items).
3. **Customer Targeting:** Focus marketing efforts on service categories that the model identifies as high-value drivers.
4. **Automation Opportunity:** Orders with high predictability can be automatically processed, while outliers (low R² segments) may need manual pricing review.